# 1.0 - Dataset and Data Split

Same shape as the study notebooks on `main`: **Connect Colab -> Download DATASET ->
Study Dataset -> Preprocess the data -> DataLoaders**. The one difference is inside
**Study Dataset**: `main` trains on the split UCF101 ships with; this notebook shows
that split leaks, replaces it with a group-disjoint split, and preprocesses that
instead.

No features are extracted anywhere. A clip is the raw normalised video frames; the
CNN is never run in this notebook.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "real-video"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "PASTE_YOUR_KAGGLE_KEY"

## **Download DATASET**

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Study Dataset**

UCF101 is a human action-recognition dataset: 13320 short videos, 101 action
classes, taken from YouTube. `DatasetStudy` reads the train / val / test CSVs the
dataset ships with and summarises the whole 101-class distribution - videos per
class, how balanced it is, the outliers - exactly as on `main`.

In [ ]:
study = DatasetStudy()
study.run_all()

### The 20 classes we use

Training all 101 classes as real video is expensive, so this study uses a 20-class
subset arranged as a short continual-learning stream:

```
Base    10 classes   (Task 0)   full-body sports, filmed far from the camera
Task 1   5 classes               close-up face / hand actions
Task 2   5 classes               two near-identical boxing actions + more
                                 10 -> 15 -> 20
```

Task 1 and Task 2 hold the classes a model tends to confuse - ApplyEyeMakeup vs
ApplyLipstick, BoxingPunchingBag vs BoxingSpeedBag - so the forgetting in the later
notebooks is visible. The lists live in `src/config/config.py`.

In [ ]:
cfg.SELECTED_CLASSES

In [ ]:
print(cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2)

In [ ]:
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx     = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}

print(f"total classes: {len(ALL_CLASSES_LIST)}")
print(f"class_to_idx:\n{class_to_idx}")

### The split the dataset ships with leaks

A UCF101 file is named like this:

```
v_ApplyEyeMakeup_g08_c03.avi
  |    class     |grp|clip
```

The `g08` field is the **group**. Every clip that shares a group number was cut from
the SAME source video - same person, same background, same lighting - and differs
only in which few seconds were taken (`c01`, `c02`, ...). Clips inside one group are
near-duplicates of each other.

The split UCF101 ships with was made **per clip, at random**, ignoring the group. So
`g08_c01` can land in train while `g08_c03` lands in test. When that happens the
model trains on frames almost identical to the ones it is later tested on: the test
score then measures memorisation of specific videos, not recognition of the action,
and every accuracy comes out inflated. That is data leakage.

`check_group_leakage` counts, for our 20 classes, how many groups the shipped split
scattered across more than one split.

In [ ]:
leaking = check_group_leakage(ALL_CLASSES_LIST)

To see it directly, here is one class laid out group by group. The rows flagged
`group_leaks = True` are clips of ONE source video that the shipped split placed in
different splits - the leakage, concretely.

In [ ]:
example  = leakage_example(cfg.SELECTED_CLASSES[0])
n_groups = example["group"].nunique()
n_leak   = example[example["group_leaks"]]["group"].nunique()

print(f"{cfg.SELECTED_CLASSES[0]}: {n_groups} groups, {n_leak} leak across splits")
example.head(15)

### A group-disjoint split fixes it

The fix is to split by **group**, not by clip: assign every whole group to exactly
one of train / val / test, so no source video is ever in two splits at once.
`build_group_split` pools all the videos, groups them by their `gXX` key, sorts the
groups by name and cuts by index - deterministic, no seed. UCF101 has 25 groups per
class, so 18 / 3 / 4 groups gives about 72 / 12 / 16 percent train / val / test.

In [ ]:
base_split  = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)
task1_split = build_group_split(cfg.TASK_1,           train_groups=18, val_groups=3)
task2_split = build_group_split(cfg.TASK_2,           train_groups=18, val_groups=3)

Clip counts per split, and the leakage check on the new split. Every line must print `groups in >1 split: 0` and `OK - group-disjoint`.

In [ ]:
clashes_base  = describe_group_split(base_split,  cfg.SELECTED_CLASSES)
clashes_task1 = describe_group_split(task1_split, cfg.TASK_1)
clashes_task2 = describe_group_split(task2_split, cfg.TASK_2)

### The comparison

```
shipped split         groups scattered across splits    ->  leakage, inflated accuracy
group-disjoint split  0 groups in more than one split   ->  honest accuracy
```

Everything from here on uses the group-disjoint split. Its accuracy numbers come out
lower than a leaky split would report - that is the correction, not a regression. The
next notebook trains on these clips and shows the honest numbers.

## **Preprocess the data**

Run these only if all three checks above said `OK - group-disjoint`.

Preprocessing turns each source video into one 16-frame clip of shape
`[16, 3, 224, 224]`: read frames at 25 fps, take 16, resize to 256, centre-crop to
224, normalise with ImageNet statistics. `preprocess_group_split` writes the clips
as `.pt` tensors to the Colab LOCAL disk (under `/content/UCF101`), NOT to Drive -
the full set is tens of gigabytes. They stay in this Colab session and the next
notebooks load them from there; if the runtime restarts, re-run this section.

In [ ]:
# BASE
preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT)

In [ ]:
# TASK 1
preprocess_group_split(task1_split, cfg.TASK_1, output_root=cfg.TASK1_ROOT)

In [ ]:
# TASK 2
preprocess_group_split(task2_split, cfg.TASK_2, output_root=cfg.TASK2_ROOT)

## **DataLoaders**

Build the loaders with `UCF101Clips` and `DataLoader`, exactly as the training
notebooks do, so this notebook doubles as the check that the clips actually load.
`class_to_idx` is the global map built above, so every class keeps the same label in
every task.

In [ ]:
base_train  = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "train"), class_to_idx=class_to_idx)
base_val    = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "val"),   class_to_idx=class_to_idx)
base_test   = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "test"),  class_to_idx=class_to_idx)

task1_train = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "train"), class_to_idx=class_to_idx)
task1_val   = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "val"),   class_to_idx=class_to_idx)
task1_test  = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "test"),  class_to_idx=class_to_idx)

task2_train = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "train"), class_to_idx=class_to_idx)
task2_val   = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "val"),   class_to_idx=class_to_idx)
task2_test  = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "test"),  class_to_idx=class_to_idx)

base_test_loader     = DataLoader(base_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task1_test_loader    = DataLoader(task1_test, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task2_test_loader    = DataLoader(task2_test, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

combined_test        = ConcatDataset([base_test, task1_test, task2_test])
combined_test_loader = DataLoader(combined_test, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

Clip counts per split, then one frame from a base clip as a sanity check that the tensors are real, normalised video.

In [ ]:
print(f"{'split':<8}{'Base':>8}{'Task1':>8}{'Task2':>8}")
print(f"{'train':<8}{len(base_train):>8}{len(task1_train):>8}{len(task2_train):>8}")
print(f"{'val':<8}{len(base_val):>8}{len(task1_val):>8}{len(task2_val):>8}")
print(f"{'test':<8}{len(base_test):>8}{len(task1_test):>8}{len(task2_test):>8}")

In [ ]:
sample_loader = DataLoader(base_train, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=2)
clips, labels = next(iter(sample_loader))

print(f"batch clips: {tuple(clips.shape)}   labels: {labels.tolist()}")
UCF101Clips.show_frame(clips, labels, clip_idx=0, frame_idx=0)